# ĐỒ ÁN MÔN HỌC: XỬ LÝ ẢNH — NHẬN DIỆN CHẤT LƯỢNG GẠO
## Phân hệ 1: Tổ chức Tệp dữ liệu và Tiền xử lý ảnh - Dúng Đố

**Đặc tả kỹ thuật:**
* **Chuẩn hóa không gian (`cv2.resize`):** Đưa toàn bộ ảnh về kích thước chuẩn `800x800` pixels bằng phép nội suy `INTER_AREA` nhằm loại bỏ răng cưa khi thu nhỏ ảnh độ phân giải cao.
* **Khử nhiễu (`cv2.GaussianBlur`):** Làm mịn nhiễu tần số cao như bụi nhỏ trên nền, nhiễu cảm biến bằng nhân lọc Gaussian `5x5`.
* **Chuyển đổi không gian màu (`cv2.cvtColor`):** Tách kênh độ sáng thang xám Grayscale phục vụ phân đoạn hình thái học.
* **Cải thiện độ tương phản (`cv2.equalizeHist`):** Cân bằng lược đồ mức xám toàn cục, kéo giãn dải động cường độ sáng giữa hạt gạo và nền tối.
* **Đóng gói dữ liệu đa nhánh:** Xuất song song dữ liệu ảnh màu sạch (cho Thành viên 3: Tuân), ảnh xám tăng cường tương phản (cho Thành viên 2: Huy) và bảng nhãn tính trạng `dataset_metadata.csv` (cho Thành viên 4: Thắng).

In [ ]:
# %% [code]
import os
import cv2
import csv
import shutil
import numpy as np

# ==============================================================================
# CẤU HÌNH ĐƯỜNG DẪN TƯƠNG ĐỐI (DÙNG CHUNG CHO MỌI MÁY TÍNH TRONG NHÓM)
# ==============================================================================
# Lấy thư mục hiện tại chứa file Jupyter Notebook làm gốc (Workspace Root)
# Dù đồng đội của bạn để project ở ổ C, ổ D hay Desktop, code vẫn tự nhận diện đúng
try:
    PROJECT_ROOT = os.path.dirname(os.path.abspath('__file__'))
except NameError:
    PROJECT_ROOT = os.getcwd() # Tương thích với môi trường VS Code Jupyter

# Định nghĩa các thư mục dựa trên gốc của Project
LABELED_DATASET_DIR = os.path.join(PROJECT_ROOT, "raw_dataset")
OUTPUT_ROOT_DIR     = os.path.join(PROJECT_ROOT, "processed_dataset")

BATCH_RICE_DIR      = os.path.join(PROJECT_ROOT, "batch_dataset") 

# Đường dẫn đầu ra Phần 1 (Reference Labeled)
OUT_REF_COLOR = os.path.join(OUTPUT_ROOT_DIR, "1_reference_labeled", "color_clean")
OUT_REF_GRAY  = os.path.join(OUTPUT_ROOT_DIR, "1_reference_labeled", "gray_enhanced")
REF_CSV_PATH  = os.path.join(OUTPUT_ROOT_DIR, "1_reference_labeled", "reference_labels.csv")

# Đường dẫn đầu ra Phần 2 (Batch Rice Images)
OUT_BATCH_COLOR = os.path.join(OUTPUT_ROOT_DIR, "2_batch_rice_images", "color_clean")
OUT_BATCH_GRAY  = os.path.join(OUTPUT_ROOT_DIR, "2_batch_rice_images", "gray_enhanced")
BATCH_CSV_PATH  = os.path.join(OUTPUT_ROOT_DIR, "2_batch_rice_images", "batch_images_list.csv")

LABEL_MAPPING = {
    "Tot": "Tốt",
    "Vo": "Vỡ",
    "Te": "Tệ",
    "Kem": "Kém",
    "Tap_chat": "Tạp chất"
}

STANDARD_SIZE = (800, 800)
SUPPORTED_EXTS = ('.jpg', '.jpeg', '.png', '.bmp')

print(f"Thư mục gốc dự án được xác định tự động tại: {PROJECT_ROOT}")

In [10]:
# %% [code]
def apply_clahe_safe(img: np.ndarray, clip_limit: float = 2.5, grid_size: tuple[int, int] = (8, 8)) -> np.ndarray:
    """
    Áp dụng CLAHE an toàn cho cả ảnh xám (2D) và ảnh màu BGR (3D).
    Với ảnh màu, chỉ áp dụng CLAHE trên kênh L (LAB) để hạn chế lệch màu.
    """
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=grid_size)
    if img.ndim == 2:
        return clahe.apply(img)

    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l_chan, a_chan, b_chan = cv2.split(lab)
    l_eq = clahe.apply(l_chan)
    lab_eq = cv2.merge((l_eq, a_chan, b_chan))
    return cv2.cvtColor(lab_eq, cv2.COLOR_LAB2BGR)


def frequency_adjust_for_grains(
    img: np.ndarray,
    low_freq_kernel: int = 51,
    low_freq_scale: float = 0.65,
    high_freq_boost: float = 1.35,
) -> np.ndarray:
    """
    Tách thành phần tần số thấp (nền / ánh sáng loang rộng) và tần số cao (biên hạt).
    Hạ bớt nền sáng loang và tăng biên hạt để rõ nét hơn dưới ánh sáng mạnh.
    """
    k = max(21, int(low_freq_kernel))
    if k % 2 == 0:
        k += 1

    img_f = img.astype(np.float32)
    low_freq = cv2.GaussianBlur(img_f, (k, k), 0)
    high_freq = img_f - low_freq

    out_f = low_freq * low_freq_scale + high_freq * high_freq_boost
    return np.clip(out_f, 0, 255).astype(np.uint8)


def process_image_dual_branch(image_path: str):
    """
    Thực thi tiền xử lý 2 nhánh cho 1 ảnh đầu vào:
    - Nhánh 1 (color_clean): Resize + khử nhiễu + cân bằng sáng nhẹ bảo toàn màu.
    - Nhánh 2 (gray_enhanced): Resize + Grayscale + giảm loang sáng + CLAHE cục bộ.
    """
    img_raw = cv2.imread(image_path)
    if img_raw is None:
        raise IOError(f"Không thể đọc tệp ảnh: {image_path}")

    # 1. Chuẩn hóa kích thước
    img_resized = cv2.resize(img_raw, STANDARD_SIZE, interpolation=cv2.INTER_AREA)

    # 2. Nhánh màu: làm sạch nhiễu nhẹ và giảm loang sáng nhưng giữ màu tự nhiên
    img_color_denoise = cv2.bilateralFilter(img_resized, d=5, sigmaColor=35, sigmaSpace=35)
    img_color_clean = apply_clahe_safe(img_color_denoise, clip_limit=1.8, grid_size=(8, 8))

    # 3. Nhánh xám: khử nhiễu giữ biên + hạ nền tần số thấp + CLAHE cục bộ chống cháy sáng
    img_gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)
    img_gray_denoise = cv2.bilateralFilter(img_gray, d=5, sigmaColor=40, sigmaSpace=40)
    img_gray_freq = frequency_adjust_for_grains(
        img_gray_denoise,
        low_freq_kernel=51,
        low_freq_scale=0.68,
        high_freq_boost=1.35,
    )
    img_gray_enhanced = apply_clahe_safe(img_gray_freq, clip_limit=3.0, grid_size=(8, 8))

    return img_color_clean, img_gray_enhanced

In [ ]:
# %% [code]
def run_labeled_reference_preprocessing():
    print("<===8 BẮT ĐẦU PHẦN 1: TIỀN XỬ LÝ NHÃN MẪU 8===>")
    
    # Tạo mới thư mục đầu ra sạch sẽ
    os.makedirs(OUT_REF_COLOR, exist_ok=True)
    os.makedirs(OUT_REF_GRAY, exist_ok=True)

    ref_records = []
    ref_total = 0

    if os.path.exists(LABELED_DATASET_DIR):
        for folder_code, vn_label in LABEL_MAPPING.items():
            src_dir = os.path.join(LABELED_DATASET_DIR, folder_code)
            if not os.path.exists(src_dir):
                continue

            dst_c_dir = os.path.join(OUT_REF_COLOR, folder_code)
            dst_g_dir = os.path.join(OUT_REF_GRAY, folder_code)
            os.makedirs(dst_c_dir, exist_ok=True)
            os.makedirs(dst_g_dir, exist_ok=True)

            # Quét file an toàn tuyệt đối, chống trùng lặp hoa/thường trên Windows
            files = []
            for fname in sorted(os.listdir(src_dir)):
                if fname.lower().endswith(SUPPORTED_EXTS):
                    files.append(os.path.join(src_dir, fname))

            print(f"  + Nhãn {vn_label:<8} ({folder_code:<8}): {len(files)} ảnh gốc")

            for idx, fpath in enumerate(files, start=1):
                try:
                    std_name = f"{folder_code}_{idx:04d}.jpg"
                    img_c, img_g = process_image_dual_branch(fpath)

                    out_c = os.path.join(dst_c_dir, std_name)
                    out_g = os.path.join(dst_g_dir, std_name)

                    cv2.imwrite(out_c, img_c, [cv2.IMWRITE_JPEG_QUALITY, 98])
                    cv2.imwrite(out_g, img_g, [cv2.IMWRITE_JPEG_QUALITY, 98])

                    ref_records.append({
                        "sample_id": f"{folder_code}_{idx:04d}",
                        "folder_code": folder_code,
                        "label_name": vn_label,
                        "original_file": os.path.basename(fpath),
                        "color_path": out_c.replace("\\", "/"),
                        "gray_path": out_g.replace("\\", "/")
                    })
                    ref_total += 1
                except Exception as e:
                    print(f"    [Lỗi] {os.path.basename(fpath)}: {e}")

        if ref_records:
            with open(REF_CSV_PATH, mode="w", newline="", encoding="utf-8-sig") as f:
                writer = csv.DictWriter(f, fieldnames=["sample_id", "folder_code", "label_name", "original_file", "color_path", "gray_path"])
                writer.writeheader()
                writer.writerows(ref_records)
                
    print(f"-> Hoàn thành Phần 1: Đã xử lý chính xác {ref_total} ảnh mẫu.\n")

run_labeled_reference_preprocessing()

In [ ]:
# %% [code]
def run_batch_rice_preprocessing():
    print("=== BẮT ĐẦU PHẦN 2: TIỀN XỬ LÝ MẺ GẠO ===")
    
    os.makedirs(OUT_BATCH_COLOR, exist_ok=True)
    os.makedirs(OUT_BATCH_GRAY, exist_ok=True)

    batch_files = []
    if os.path.exists(BATCH_RICE_DIR):
        for fname in sorted(os.listdir(BATCH_RICE_DIR)):
            if fname.lower().endswith(SUPPORTED_EXTS):
                batch_files.append(os.path.join(BATCH_RICE_DIR, fname))

    print(f"  + Tìm thấy {len(batch_files)} ảnh mẻ gạo thực tế.")
    batch_records = []
    batch_total = 0

    for idx, fpath in enumerate(batch_files, start=1):
        try:
            std_name = f"batch_rice_{idx:04d}.jpg"
            img_c, img_g = process_image_dual_branch(fpath)

            out_c = os.path.join(OUT_BATCH_COLOR, std_name)
            out_g = os.path.join(OUT_BATCH_GRAY, std_name)

            cv2.imwrite(out_c, img_c, [cv2.IMWRITE_JPEG_QUALITY, 98])
            cv2.imwrite(out_g, img_g, [cv2.IMWRITE_JPEG_QUALITY, 98])

            batch_records.append({
                "batch_id": f"batch_rice_{idx:04d}",
                "original_file": os.path.basename(fpath),
                "color_path": out_c.replace("\\", "/"),
                "gray_path": out_g.replace("\\", "/")
            })
            batch_total += 1
        except Exception as e:
            print(f"  [Lỗi] {os.path.basename(fpath)}: {e}")

    if batch_records:
        with open(BATCH_CSV_PATH, mode="w", newline="", encoding="utf-8-sig") as f:
            writer = csv.DictWriter(f, fieldnames=["batch_id", "original_file", "color_path", "gray_path"])
            writer.writeheader()
            writer.writerows(batch_records)

    print(f"-> Hoàn thành Phần 2: Đã xử lý chính xác {batch_total} ảnh mẻ gạo.")
    print("=" * 65)
    print("HOÀN TẤT TOÀN BỘ QUY TRÌNH! DỮ LIỆU ĐÃ SẠCH SẼ")
    print(f"Thư mục kết quả: {OUTPUT_ROOT_DIR}")
    print("=" * 65)

run_batch_rice_preprocessing()